In [1]:
# Imports

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from utils_data_io import tsv_to_tfds
from utils_model import AKImageClassifierAPI

C:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load dataset

In [4]:
df = pd.read_csv("styles.csv", on_bad_lines="skip")
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
# Filter classes

In [6]:
target_categories = [
    "Accessories",
    "Apparel",
    "Footwear",
    "Free Items",
    "Personal Care",
    "Sporting Goods",
]

df = df[df.masterCategory.isin(target_categories)].copy()
df["image_path"] = "images/" + df["id"].astype(str) + ".jpg"
df["label_name"] = df["masterCategory"]

class_names = sorted(df["label_name"].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}

df["label_idx"] = df["label_name"].map(class_to_idx)

class_names

['Accessories',
 'Apparel',
 'Footwear',
 'Free Items',
 'Personal Care',
 'Sporting Goods']

In [7]:
# Remove missing images

In [8]:
df = df[df["image_path"].apply(os.path.exists)].copy()
len(df)

44418

In [9]:
# Train/Val/Test split

In [10]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label_idx"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label_idx"], random_state=42
)

In [11]:
# Save TSV

In [12]:
os.makedirs("lists", exist_ok=True)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split[["image_path", "label_idx"]].to_csv(
        f"lists/{name}.tsv",
        sep="\t",
        header=False,
        index=False,
    )

In [13]:
# Build tf.data datasets

In [14]:
num_classes = len(class_names)

train_ds = tsv_to_tfds("lists/train.tsv", num_classes)
val_ds   = tsv_to_tfds("lists/val.tsv", num_classes)
test_ds  = tsv_to_tfds("lists/test.tsv", num_classes)

In [ ]:
# Load saved model and evaluate

In [16]:
import autokeras as ak

model = tf.keras.models.load_model(
    "models/autokeras_best.h5",
    custom_objects=ak.CUSTOM_OBJECTS,
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

test_loss, test_acc = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

139/139 ━━━━━━━━━━━━━━━━━━━━ 51s 275ms/step - accuracy: 0.4809 - loss: 1.1857
Test loss: 1.1856721639633179
Test accuracy: 0.4808644652366638


In [17]:
# Evaluation

In [18]:
preds = []
true = []

for x,y in test_ds:
    p = model.predict(x)
    preds.extend(np.argmax(p,axis=1))
    true.extend(y.numpy())

report = classification_report(true, preds, target_names=class_names)

os.makedirs("outputs", exist_ok=True)

with open("outputs/report_autokeras.txt","w") as f:
    f.write(report)

cm = confusion_matrix(true, preds)

plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.savefig("outputs/confmat_autokeras.png")
plt.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

C:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
